In [3]:
import yaml
import pandas as pd
from sqlalchemy import create_engine
from urllib.parse import quote_plus

# Load database config from YAML
with open("../configs/database.yaml", "r") as f:
    config = yaml.safe_load(f)

pg_conf = config["database"]["postgres"]
pg_user = quote_plus(pg_conf["user"])
pg_password = quote_plus(pg_conf["password"])
pg_host = pg_conf["host"]
pg_port = pg_conf["port"]
pg_db = pg_conf["db"]

conn_str = (
    f"postgresql+psycopg2://{pg_user}:{pg_password}@{pg_host}:{pg_port}/{pg_db}"
)
engine = create_engine(conn_str)
print("Connected to PostgreSQL:", pg_conf['host'], pg_conf['db'])

Connected to PostgreSQL: localhost storage_analytics


In [ ]:
anomaly_df = pd.read_sql_query("SELECT * FROM anomaly_events", engine)
curated_df = pd.read_sql_query("SELECT * FROM curated_device_metrics", engine)

curated_df["timestamp"] = pd.to_datetime(curated_df["timestamp"])
anomaly_df["timestamp"] = pd.to_datetime(anomaly_df["timestamp"])

anomaly_df.head()

,device,timestamp,metric_name,metric_value,detector_type,anomaly_score,severity,is_anomaly,details,source_file,...,avg_latency_ms,read_ratio,write_ratio,avg_request_size_kb,workload_pattern,total_iops,total_throughput_mb_s,saturation_score,latency_pressure,root_cause_hint
0,dm-0,2026-03-28 21:55:49.203082+00:00,avg_latency_ms,30.628903,iqr,7.118702,critical,1,"lower=-2.02, upper=9.73",data/raw/generated_iostat.csv,...,30.628903,0.527940,0.472060,46.195528,saturated,143.254767,6.462627,249.430928,214.950844,Latency anomaly detected without clear saturat...
1,dm-0,2026-03-28 22:06:19.041760+00:00,avg_latency_ms,17.566847,iqr,2.670217,medium,1,"lower=-2.02, upper=9.73",data/raw/generated_iostat.csv,...,17.566847,0.471129,0.528871,70.503327,saturated,122.737342,8.450577,359.874536,179.675111,Latency spike likely driven by saturation and ...
2,dm-0,2026-03-29 18:11:24.026588+00:00,avg_latency_ms,31.753232,iqr,7.501610,critical,1,"lower=-2.02, upper=9.73",data/raw/generated_iostat.csv,...,31.753232,0.702797,0.297203,40.949562,saturated,315.616668,12.621450,250.092405,215.600616,Latency spike observed during read-heavy demand
3,dm-0,2026-03-29 18:15:55.122781+00:00,avg_latency_ms,14.615001,iqr,1.664920,medium,1,"lower=-2.02, upper=9.73",data/raw/generated_iostat.csv,...,14.615001,0.577072,0.422928,46.345198,saturated,320.527468,14.506747,406.602199,174.592174,Latency spike likely driven by saturation and ...
4,dm-0,2026-03-29 18:26:01.172401+00:00,avg_latency_ms,33.193017,iqr,7.991951,critical,1,"lower=-2.02, upper=9.73",data/raw/generated_iostat.csv,...,33.193017,0.508158,0.491842,33.773162,saturated,407.258283,13.432031,280.656475,258.401882,Latency anomaly detected without clear saturat...


In [5]:
anomaly_df["detector_type"].value_counts()
anomaly_df["severity"].value_counts()
anomaly_df["metric_name"].value_counts()

metric_name
latency_pressure    633
multivariate        504
aqu_sz              500
avg_latency_ms      494
saturation_score    403
total_iops          114
util_pct             57
Name: count, dtype: int64

In [6]:
anomaly_df.groupby("device").size().sort_values(ascending=False)

device
sdb        1336
dm-0        503
nvme1n1     369
sda         268
nvme0n1     229
dtype: int64

In [7]:
anomaly_df.sort_values("anomaly_score", ascending=False).head(20)

,device,timestamp,metric_name,metric_value,detector_type,anomaly_score,severity,is_anomaly,details,source_file,...,avg_latency_ms,read_ratio,write_ratio,avg_request_size_kb,workload_pattern,total_iops,total_throughput_mb_s,saturation_score,latency_pressure,root_cause_hint
567,nvme0n1,2026-03-30 01:31:19.815065+00:00,latency_pressure,265.535632,iqr,2725.821295,critical,1,"lower=0.25, upper=0.64",data/raw/generated_iostat.csv,...,28.497678,0.734460,0.265540,97.359631,saturated,146.383183,13.917786,306.529323,265.535632,Joint increase in latency and queue depth sugg...
626,nvme0n1,2026-04-03 15:31:12.991458+00:00,latency_pressure,259.401118,iqr,2662.696692,critical,1,"lower=0.25, upper=0.64",data/raw/generated_iostat.csv,...,28.608941,0.638263,0.361737,134.926591,saturated,304.142485,40.075106,310.494560,259.401118,Joint increase in latency and queue depth sugg...
574,nvme0n1,2026-03-30 10:36:23.912038+00:00,latency_pressure,223.972251,iqr,2298.131002,critical,1,"lower=0.25, upper=0.64",data/raw/generated_iostat.csv,...,23.264695,0.527660,0.472340,102.146818,saturated,298.158710,29.742152,351.556267,223.972251,Joint increase in latency and queue depth sugg...
578,nvme0n1,2026-03-30 10:56:14.733657+00:00,latency_pressure,223.822530,iqr,2296.590361,critical,1,"lower=0.25, upper=0.64",data/raw/generated_iostat.csv,...,21.315515,0.643219,0.356781,74.437996,saturated,450.468175,32.746043,343.086442,223.822530,Joint increase in latency and queue depth sugg...
587,nvme0n1,2026-03-30 12:41:19.911071+00:00,latency_pressure,221.495211,iqr,2272.642074,critical,1,"lower=0.25, upper=0.64",data/raw/generated_iostat.csv,...,29.913540,0.676998,0.323002,71.133106,saturated,509.147437,35.368397,265.761587,221.495211,Joint increase in latency and queue depth sugg...
588,nvme0n1,2026-03-30 12:46:14.749276+00:00,latency_pressure,188.175886,iqr,1929.783730,critical,1,"lower=0.25, upper=0.64",data/raw/generated_iostat.csv,...,10.548185,0.707708,0.292292,100.427079,saturated,464.911451,45.595409,567.229875,188.175886,Joint increase in latency and queue depth sugg...
585,nvme0n1,2026-03-30 12:26:31.955210+00:00,latency_pressure,174.871963,iqr,1792.885363,critical,1,"lower=0.25, upper=0.64",data/raw/generated_iostat.csv,...,23.541848,0.765608,0.234392,57.964737,saturated,510.870608,28.918438,249.225211,174.871963,Joint increase in latency and queue depth sugg...
563,nvme0n1,2026-03-30 01:10:47.777136+00:00,latency_pressure,164.203560,iqr,1683.106706,critical,1,"lower=0.25, upper=0.64",data/raw/generated_iostat.csv,...,29.916850,0.681623,0.318377,54.958645,saturated,154.609640,8.297985,194.493643,164.203560,Joint increase in latency and queue depth sugg...
562,nvme0n1,2026-03-30 01:06:25.888292+00:00,latency_pressure,148.841907,iqr,1525.034149,critical,1,"lower=0.25, upper=0.64",data/raw/generated_iostat.csv,...,15.936427,0.665969,0.334031,73.772129,saturated,155.942763,11.234599,330.749024,148.841907,Joint increase in latency and queue depth sugg...
568,nvme0n1,2026-03-30 01:36:33.957845+00:00,latency_pressure,132.913668,iqr,1361.131378,critical,1,"lower=0.25, upper=0.64",data/raw/generated_iostat.csv,...,9.697778,0.727682,0.272318,87.510213,saturated,127.792842,10.921073,446.176336,132.913668,Joint increase in latency and queue depth sugg...
